In [219]:
import numpy as np
import pandas as pd
import re
from sklearn.base import BaseEstimator, TransformerMixin


In [220]:
train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')

In [221]:
def sentence_metrics(text):
    sentences = [s.strip() for s in re.split(r'[.!?]+', text) if len(s.strip()) > 0]
    if not sentences:
        return 0, 0, 0
    
    words_per_sentence = [len(s.split()) for s in sentences]
    
    num_sentences = len(sentences)
    avg_words = np.mean(words_per_sentence)
    std_words = np.std(words_per_sentence)
    
    return num_sentences, avg_words, std_words

# humans often alternate long sentences with shorter ones... (I hope so)
sentence_metrics(train['TEXT'][0]), train['LABEL'][0]

((31, np.float64(16.870967741935484), np.float64(7.079010025027744)),
 np.int64(0))

In [222]:
import re
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin

class featureExtractor(BaseEstimator, TransformerMixin):
     def __init__(self):
          pass

     # humans often alternate long sentences with shorter ones... (I hope so)
     @staticmethod
     def __sentence_metrics(text):
          sentences = [s.strip() for s in re.split(r'[.!?]+', text) if len(s.strip()) > 0]
          if not sentences:
               return 0, 0, 0
          
          words_per_sentence = [len(s.split()) for s in sentences]
          
          num_sentences = len(sentences)
          avg_words = np.mean(words_per_sentence)
          std_words = np.std(words_per_sentence)
          return num_sentences, avg_words, std_words

     def fit(self, X, y=None):
          return self
     
     def transform(self, X):
          df = X.copy()
          
          df['TEXT_str'] = df['TEXT'].astype(str) 
          df['len'] = df['TEXT_str'].apply(len)
          df['words_list'] = df['TEXT_str'].str.split()
          
          df['num_words'] = df['words_list'].apply(lambda x: len(x) if isinstance(x, list) else 0)
          
          # avoiding to explode something in case of missing text
          df['len_safe'] = df['len'].replace(0, 1)
          df['words_safe'] = df['num_words'].replace(0, 1)

          # Punctuation...
          punctuation_chars = {
               'periods': '.', 'commas': ',', 'dashes': '-', 
               'question': '?', 'exclamation': '!', 'semicolon': ';', 'colon': ':',
               'spaces': ' ', 'newlines': '\n', 'asterisks': '*'
          }

          for name, char in punctuation_chars.items():
               df[name] = df['TEXT_str'].apply(lambda x: x.count(char))
               df[f'{name}_per_len'] = df[name] / df['len_safe']

          # mixed case like parenthesis and quotes
          df['parenthesis'] = df['TEXT_str'].apply(lambda x: x.count('(') + x.count(')'))
          df['parenthesis_per_len'] = df['parenthesis'] / df['len_safe']

          df['quotes'] = df['TEXT_str'].apply(lambda x: x.count('"') + x.count("'") + x.count('`'))
          df['quotes_per_len'] = df['quotes'] / df['len_safe']

          # uppercase and digits
          df['uppercase'] = df['TEXT_str'].apply(lambda x: len(re.findall(r'[A-Z]', x)))
          df['uppercase_per_len'] = df['uppercase'] / df['len_safe']

          df['digits'] = df['TEXT_str'].apply(lambda x: len(re.findall(r'\d', x)))
          df['digits_per_len'] = df['digits'] / df['len_safe']

          # unique words
          df['unique_words'] = df['words_list'].apply(lambda x: len(set(x)) if isinstance(x, list) else 0)
          df['unique_words_per_words'] = df['unique_words'] / df['words_safe']

          df['avg_word_length'] = df['words_list'].apply(
               lambda x: np.mean([len(w) for w in x]) if isinstance(x, list) and len(x) > 0 else 0
          )

          # enumerations
          df['enumeration_num'] = df['TEXT_str'].apply(lambda x: len(re.findall(r'\d+\.', x)))
          df['enumeration_num_per_len'] = df['enumeration_num'] / df['len_safe']

          metrics = df['TEXT_str'].apply(self.__sentence_metrics)
          df['num_sentences'] = [m[0] for m in metrics]
          df['sentences_per_len'] = df['num_sentences'] / df['len_safe']
          df['avg_words_per_sentence'] = [m[1] for m in metrics]
          df['std_sentence_length'] = [m[2] for m in metrics]

          # Cleaning
          df = df.drop(columns=['TEXT_str', 'words_list', 'len_safe', 'words_safe'])
          
          return df
     
     def fit_transform(self, X, y = None, **fit_params):
          return super().fit_transform(X, y, **fit_params)

In [223]:
from sklearn.model_selection import train_test_split
df_train, df_test = train_test_split(train, test_size=0.2, stratify=train['LABEL'], random_state=42)
X_train, y_train = df_train['TEXT'], df_train['LABEL']
X_test, y_test = df_test['TEXT'], df_test['LABEL']

In [224]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import chi2

class Chi2TextFeatureSelector(BaseEstimator, TransformerMixin):
     """
     This class applies TF-IDF (Term Frequency-Inverse Document Frequency) to text data
     and selects the most important features (words/n-grams) for each class using the 
     Chi-Square (Chi2) statistical test. It uses a One-vs-Rest strategy. (It was also applied in the DSMLL course by me)
     """
     
     def __init__(self, 
                    text_col: str = 'TEXT', 
                    k_per_label: int = 50, 
                    min_df: int = 3,  
                    ngram_range: tuple = (1, 2)):
          
          self.text_col = text_col
          # k_per_label: How many top features to select for each unique class.
          self.k_per_label = k_per_label
          # min_df: Minimum Document Frequency. Ignores words that appear in fewer than 'min_df' documents.
          self.min_df = min_df
          # ngram_range: (1, 2) means we extract single words (unigrams) and two-word phrases (bigrams).
          self.ngram_range = ngram_range
          
          # Attributes that will be learned during the fit()
          self.tfidf_vectorizer_ = None
          self.selected_indices_ = None 
          self.feature_names_ = None
          
     def fit(self, X: pd.DataFrame, y) -> 'Chi2TextFeatureSelector':
          """
          Learns the vocabulary from the text and selects the best features based on the Chi2 test.
          """
          # I absolutely need the labels to compute the Chi-Square statistical test!
          if y is None:
               raise ValueError("Target variable 'y' is required to compute Chi2.")
          
          y_arr = y.values if isinstance(y, pd.Series) else np.array(y)
          
          text_data = X[self.text_col].fillna('').astype(str)
          
          print(f"Fitting TF-IDF (min_df={self.min_df}, ngrams={self.ngram_range})...")
          
          self.tfidf_vectorizer_ = TfidfVectorizer(
               input='content', encoding='utf-8', lowercase=True,
               stop_words=None, # I want to keep stop words because they might be important for classification (e.g., "not", "but", "and") 
               min_df=self.min_df, 
               ngram_range=self.ngram_range
          )
          
         
          # Transform the text into a sparse matrix of TF-IDF scores
          X_tfidf = self.tfidf_vectorizer_.fit_transform(text_data)
          print(f"Selecting top {self.k_per_label} features per label via Chi-Square test...")
          unique_classes = np.unique(y_arr)
          
          feature_to_labels = {}

          # Computing Chi-Square for each class using the "One-vs-Rest" approach
          for label in unique_classes:
               # 1 if it's the current class, 0 otherwise
               y_binary = (y_arr == label).astype(int)
               
               # Compute the Chi2 scores between all TF-IDF features and the binary target
               # The chi2() function returns two arrays: scores and p-values. I only care about the scores.
               chi2_scores, _ = chi2(X_tfidf, y_binary)
               
               # Get the total number of available features in the TF-IDF vocabulary
               n_features = X_tfidf.shape[1]
               
               # Ensure we don't try to select more features than actually exist
               k_safe = min(self.k_per_label, n_features)

               if k_safe > 0:
                    top_k_indices = np.argsort(chi2_scores)[-k_safe:]
                    for idx in top_k_indices:
                         if idx not in feature_to_labels:
                              feature_to_labels[idx] = set()
                              # Record that this specific word index is a strong predictor for the current 'label'
                         feature_to_labels[idx].add(label)

          # Finalizing the selected indices and create descriptive column names
          # Extract all unique indices selected across all classes and sort them
          self.selected_indices_ = sorted(list(feature_to_labels.keys()))
          
          if not self.selected_indices_:
               print("Warning: No features were selected.")
               self.feature_names_ = []
               return self
               
          # Get the actual string words/n-grams from the fitted TF-IDF vocabulary
          raw_feature_names = self.tfidf_vectorizer_.get_feature_names_out()
          self.feature_names_ = []
          
          for idx in self.selected_indices_:
               # Extract the actual word corresponding to the numerical index
               word = raw_feature_names[idx]
               
               # Create a string suffix of the labels that this word helps predict (e.g., "0_3")
               labels_suffix = "_".join(sorted([str(lbl) for lbl in feature_to_labels[idx]]))
               
               # Construct the final descriptive feature name (e.g., "tfidf_apple_L0_3")
               self.feature_names_.append(f"tfidf_{word}_L{labels_suffix}")
               
          print(f"Total unique TF-IDF features selected: {len(self.selected_indices_)}")
          return self
     
     def transform(self, X: pd.DataFrame) -> pd.DataFrame:
          """
          Applies the learned TF-IDF transformation and filters the matrix to keep 
          only the features selected by the Chi2 test during the fit() phase.
          """
          # Check if the model has been fitted properly
          if self.tfidf_vectorizer_ is None:
               raise RuntimeError("The Transformer is not fitted yet. Call fit() before transform().")

          # Create a copy to avoid altering the original dataframe in memory
          df = X.copy()
          text_data = df[self.text_col].fillna('').astype(str)
          
          # Initialize a list of dataframes to concatenate at the end.
          # We start by removing the original raw text column so it doesn't get passed to the ML model.
          dfs_to_concat = [df.drop(columns=[self.text_col], errors='ignore')]
          
          if len(self.selected_indices_) > 0:
               # Transform the new text data using the ENTIRE vocabulary learned during fit()
               X_tfidf_full = self.tfidf_vectorizer_.transform(text_data)
               
               # Feature Selection (Filtering)
               # Slice the sparse matrix: keep ONLY the columns (indices) selected by the Chi2 test
               X_tfidf_sel = X_tfidf_full[:, self.selected_indices_]
               
               # Convert the sparse matrix into a dense Pandas DataFrame
               df_tfidf = pd.DataFrame(
                    X_tfidf_sel.toarray(),         # .toarray() converts the sparse matrix to a dense NumPy array
                    columns=self.feature_names_,   # Apply the descriptive column names we generated in fit()
                    index=df.index                 # Keep the original index to ensure alignment with other features
               )
               
               # Add the new TF-IDF numerical dataframe to our concatenation list
               dfs_to_concat.append(df_tfidf)
               
          # Concatenate horizontally (axis=1) to combine any existing features with the new text features
          final_df = pd.concat(dfs_to_concat, axis=1)
          return final_df

In [225]:
X_train = pd.DataFrame(X_train)

In [226]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pipeline = Pipeline(
     [
          
          ('feature_ext', featureExtractor()),
          ('impChi', Chi2TextFeatureSelector(k_per_label=45, min_df=5, ngram_range=(1, 2))),
          ('scaler', StandardScaler()) 
     ]
)

X_train_pp = pipeline.fit_transform(pd.DataFrame(X_train), y_train)
X_test_pp = pipeline.transform(pd.DataFrame(X_test))

Fitting TF-IDF (min_df=5, ngrams=(1, 2))...
Selecting top 45 features per label via Chi-Square test...
Total unique TF-IDF features selected: 252


In [227]:
from lightgbm import LGBMClassifier
best_params = {
    'objective': 'multiclass',
    'num_class': 6,
    'class_weight': 'balanced',
    'boosting_type': 'gbdt',
    'random_state': 42,
    'n_jobs': -1,
    'verbosity': -1,

    # Optimized parameters by Optuna and StratifiedKFold cross-validation
    'n_estimators': 400,
    'learning_rate': 0.0235,
    'num_leaves': 248,
    'max_depth': 12,
    'min_child_samples': 67,
    'reg_alpha': 0.215,
    'reg_lambda': 1.204,
    'min_split_gain': 0.044,
    'colsample_bytree': 0.337,
    'subsample': 0.887,
    'subsample_freq': 4
}

model = LGBMClassifier(**best_params)

In [228]:
from sklearn.metrics import classification_report


model.fit(X_train_pp, y_train)
y_pred = model.predict(X_test_pp)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       304
           1       0.81      0.81      0.81        16
           2       0.97      0.88      0.92        32
           3       1.00      1.00      1.00        16
           4       0.96      1.00      0.98        48
           5       0.97      0.98      0.98        64

    accuracy                           0.98       480
   macro avg       0.95      0.94      0.95       480
weighted avg       0.98      0.98      0.98       480



c:\Users\gianb\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
